In [1]:
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error
from itertools import product

Progress:
- Extract data for each grid
- Find the best hyperparameters for each grid
- Save the best hyperparameters for each grid

In [85]:
def extract_grid_data(df, grid):
    grid_data = df[(df["x_coord"] == grid[0]) & (df["y_coord"] == grid[1])].copy()
    grid_data = grid_data[["date_time", "water_percentage"]]
    grid_data["date_time"] = pd.to_datetime(grid_data["date_time"])
    grid_data = grid_data.set_index("date_time").sort_index()
    grid_data = grid_data[["water_percentage"]].reset_index(drop=True)
    return grid_data

In [86]:
def rolling_one_step_predictions(model_fit, y_val):
    rolling_fit = model_fit
    predictions = []

    for actual_value in y_val:
        pred = rolling_fit.get_forecast(steps=1).predicted_mean.iloc[0]
        pred = max(pred, 0)
        predictions.append(pred)
        new_observation = pd.DataFrame(
            {"water_percentage": [actual_value]},
            index=[rolling_fit.nobs]
        )
        rolling_fit = rolling_fit.append(
            new_observation,
            refit=False
        )
    return np.array(predictions)

In [87]:
def get_best_param(train_data, val_data):
    # Initialize the variables
    best_mse = float("inf")
    best_order = None
    best_seasonal_order = None
    
    y_train = train_data["water_percentage"]
    y_val = val_data["water_percentage"]
    
    # Define ranges for p and q values
    p_range, q_range = range(0, 3), range(0, 3)
    P_range, q_range = range(0, 2), range(0, 2)
    d_range, D_range = range(0, 2), range(0, 2) 

    # Nested loops
    for p, d, q, P, D, Q in product(
        p_range, d_range, q_range,
        P_range, D_range, q_range
    ):
        try: 
            model = SARIMAX(y_train, 
                            order=(p, 1, q), 
                            seasonal_order=(P, 1, Q, 27),
                            enforce_stationarity=False, 
                            enforce_invertibility=False)
            
            fit = model.fit(disp=False, maxiter=50)
            
            # Predict and calculate MSE
            predictions = rolling_one_step_predictions(fit, y_val)
            mse = mean_squared_error(y_val, predictions)
            
            # Update if this is the lowest MSE
            if mse < best_mse:
                best_mse = mse
                best_order = (p, 1, q)
                best_seasonal_order = (P, 1, Q, 27)
        except Exception as e:
            print("Failed:", (p, d, q), (P, D, Q, 27), "Error:", e)
            continue # Skip failed convergence

    return best_order, best_seasonal_order

In [4]:
df = pd.read_csv("../data/04_processed/train.csv")
train_df = df[(df["year"] >= 2000) & (df["year"] < 2013)]
val_df = df[(df["year"] >= 2013) & (df["year"] < 2017)]

In [89]:
grid_list = df[["x_coord", "y_coord"]].drop_duplicates().values.tolist()

In [90]:
import warnings
from joblib import Parallel, delayed
from statsmodels.tools.sm_exceptions import ValueWarning, ConvergenceWarning

def process_grid(grid):
    # Ignore warnings
    warnings.filterwarnings("ignore", category=ValueWarning)
    warnings.filterwarnings("ignore", category=FutureWarning)
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    warnings.filterwarnings("ignore", module="statsmodels")
    
    train_data = extract_grid_data(train_df, grid)
    val_data = extract_grid_data(val_df, grid)
    order, seasonal_order = get_best_param(train_data, val_data)
    
    return {
        "grid": grid, 
        "order": order, 
        "seasonal_order": seasonal_order
    }

In [91]:
# Run the parallel search
results_list = Parallel(n_jobs=4)(delayed(process_grid)(grid) for grid in grid_list)
results_df = pd.DataFrame(results_list)

Failed: (2, 0, 1) (0, 0, 1, 27) Error: Input contains NaN.
Failed: (2, 0, 1) (0, 1, 1, 27) Error: Input contains NaN.
Failed: (2, 1, 1) (0, 0, 1, 27) Error: Input contains NaN.
Failed: (2, 1, 1) (0, 1, 1, 27) Error: Input contains NaN.


Exception ignored in: <function ResourceTracker.__del__ at 0x10496dbc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes


In [94]:
results_df[["order", "seasonal_order"]].drop_duplicates()

,order,seasonal_order
0,"(1, 1, 1)","(1, 1, 1, 27)"
1,"(1, 1, 1)","(0, 1, 1, 27)"
2,"(2, 1, 1)","(0, 1, 1, 27)"
13,"(0, 1, 1)","(0, 1, 1, 27)"
16,"(1, 1, 1)","(1, 1, 0, 27)"
20,"(2, 1, 1)","(1, 1, 0, 27)"
23,"(2, 1, 1)","(1, 1, 1, 27)"
35,"(0, 1, 0)","(0, 1, 1, 27)"
37,"(0, 1, 1)","(1, 1, 1, 27)"
59,"(0, 1, 0)","(1, 1, 1, 27)"


In [ ]:
results_df.to_csv("sarima_parameters.csv")

Exception ignored in: <function ResourceTracker.__del__ at 0x1055bdbc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x106acdbc0>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x1050edbc0>
Traceback (most recent call last